Now that you've deployed your endpoint - it's time to slam it!

In [ ]:
import os
import getpass
from dotenv import load_dotenv

# override=True so values in .env (the Fireworks key, LangSmith tracing vars, etc.)
# win over any empty/stale variables already in the shell environment.
load_dotenv(override=True)

if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter your Fireworks API key: ")

Let's try with 1 request, just to verify our endpoint is alive.

Make sure you provide your own endpoint identifier! It will look something like this:

- `accounts/fireworks/models/gpt-oss-20b` (serverless)
- Or your custom on-demand deployment identifier

In [9]:
import time

from langchain_fireworks import ChatFireworks

model_endpoint = "accounts/fireworks/models/gpt-oss-20b"

llm = ChatFireworks(model=model_endpoint)

start = time.perf_counter()
response = llm.invoke("How much wood could a wood chuck chuck if a wood chuck could chuck wood?")
elapsed = time.perf_counter() - start

print(response.content)
print(f"\nSingle request latency: {round(elapsed, 1)}s")

Ah, the age‑old riddle that’s been making grins since the 1930s! 😄

**The “classic” answer**  
In folklore, it’s usually said that a woodchuck (also known as a *groundhog*) would chuck **about 700 pounds (roughly 320 kg)** of wood — which works out to roughly **1.3 cubic feet** of wood if you average a log to about 3 inches thick. That’s about the volume a typical ground‑hog could push with its stout paws.

**Why the number?**  
The 700‑pound figure comes from a 1988 *New Scientist* article that used the chucking frequency of a woodchuck in burrow construction to back‑of‑the‑envelope compute how much material it could move. Of course, it’s all model‑based and the actual ground‑hog weight is far less than a human, so it’s more a tongue‑twister delight than mountain‑science.

**Bottom line**  
There isn’t a single “official” answer—but the conventional humor answer is:

> **A woodchuck would chuck around 700 pounds of wood — if it could!**  

Feel free to use that answer in your next con

Now, let's SLAM IT.

In [10]:
import asyncio
import time

async def send_request(llm, idx):
    start = time.perf_counter()
    try:
        response = await llm.ainvoke(
            f"How much wood could a wood chuck chuck if a wood chuck could chuck wood? (Request {idx})"
        )
        elapsed = time.perf_counter() - start
        print(f"Response {idx}: {response.content[:60]}... ({round(elapsed, 1)}s)")
        return elapsed
    except Exception as e:
        print(f"Request {idx} failed: {e}")
        return None

async def main():
    num_requests = 24

    batch_start = time.perf_counter()
    tasks = [send_request(llm, i) for i in range(1, num_requests + 1)]
    latencies = await asyncio.gather(*tasks)
    total_time = time.perf_counter() - batch_start

    # Keep only the requests that succeeded.
    ok = [seconds for seconds in latencies if seconds is not None]

    print("\n===== LOAD TEST RESULTS =====")
    print(f"Concurrent requests: {num_requests}")
    print(f"Succeeded: {len(ok)} / {num_requests}")
    print(f"Total time for the batch: {round(total_time, 1)}s")
    if ok:
        average_latency = sum(ok) / len(ok)
        print(f"Average request latency: {round(average_latency, 1)}s")
        print(f"Fastest request: {round(min(ok), 1)}s")
        print(f"Slowest request: {round(max(ok), 1)}s")
        print(f"Throughput: {round(num_requests / total_time, 1)} requests/sec")

await main()

Response 9: Sure, here's a short answer: Quoting a 1973 study, a woodchu... (4.1s)
Response 3: The classic tongue‑twister answer usually goes: **“A woodchu... (5.0s)
Response 2: The old tongue‑twister has no one exact answer—it's meant mo... (6.6s)
Response 4: In the tongue‑twister world it’s really a classic play on wo... (8.5s)
Response 23: **Answer**

A classic tongue‑twister, but scientists have ta... (8.5s)
Response 13: **Answer:**  
A wood‑chuck could chuck about **700 pounds** ... (8.7s)
Response 1: **Answer (tongue‑twister style)**  
If a woodchuck could chu... (11.3s)
Response 15:    A classic tongue‑twister, so the answer is usually given ... (11.8s)
Response 11: Ah, the classic tongue‑twister that’s more about rhythm than... (12.6s)
Response 6: In the classic tongue‑twister the answer is a bit of playful... (13.0s)
Response 10: **Answer to the classic tongue‑twister**

> “How much wood c... (13.1s)
Response 21: **Answer: “Just—** *As much wood as a wood‑chuck would chuck... 